# Lesson 5b: PCA Practical — Kernel PCA and Face Recognition

## Overview
This lesson applies PCA and kernel PCA to real-world data. We'll use face images to demonstrate Eigenfaces, measure reconstruction quality, and explore how PCA serves as a feature extraction method.

**Learning Goals:**
- Use scikit-learn PCA API
- Standardize data and understand scaling
- Implement kernel PCA for nonlinear reduction
- Eigenfaces: PCA on face images
- Measure reconstruction error vs components
- PCA as preprocessing for classification

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_lfw_people, load_iris
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Part 1: scikit-learn PCA Workflow

Standard workflow: fit on training data, transform train/test.

In [ ]:
# Use Iris dataset as simple example
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target

# Standardize (important!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Original shape: {X.shape}")
print(f"PCA shape (2 components): {X_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.sum():.1%}")

# Visualize
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
for target in np.unique(y):
    plt.scatter(X_pca[y == target, 0], X_pca[y == target, 1], 
               label=iris.target_names[target], alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('PCA of Iris Dataset')
plt.legend()

# Scree plot
plt.subplot(1, 2, 2)
pca_all = PCA()
pca_all.fit(X_scaled)
plt.plot(np.cumsum(pca_all.explained_variance_ratio_), 'bo-')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Scree Plot')
plt.grid(True)
plt.tight_layout()
plt.show()

## Part 2: Importance of Standardization

PCA is sensitive to scale. Always standardize unless dimensions are on the same scale.

In [ ]:
# Generate data with different scales
np.random.seed(42)
X_scaled_data = np.random.randn(100, 2)
X_scaled_data[:, 0] *= 1000  # Scale first dimension
X_scaled_data[:, 1] *= 1     # Keep second dimension

# PCA without scaling
pca_no_scale = PCA(n_components=2)
X_no_scale = pca_no_scale.fit_transform(X_scaled_data)

# PCA with scaling
scaler = StandardScaler()
X_scaled_data_norm = scaler.fit_transform(X_scaled_data)
pca_with_scale = PCA(n_components=2)
X_with_scale = pca_with_scale.fit_transform(X_scaled_data_norm)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original unscaled
axes[0, 0].scatter(X_scaled_data[:, 0], X_scaled_data[:, 1])
axes[0, 0].set_title('Original (Unscaled)')
axes[0, 0].set_xlabel('x₀ (scale ~1000)')
axes[0, 0].set_ylabel('x₁ (scale ~1)')

# Original scaled
axes[0, 1].scatter(X_scaled_data_norm[:, 0], X_scaled_data_norm[:, 1])
axes[0, 1].set_title('After StandardScaler')
axes[0, 1].set_xlim(-3, 3)
axes[0, 1].set_ylim(-3, 3)

# PCA without scaling
axes[1, 0].scatter(X_no_scale[:, 0], X_no_scale[:, 1])
axes[1, 0].set_title(f'PCA without Scaling\nPC1 explains {pca_no_scale.explained_variance_ratio_[0]:.1%}')
axes[1, 0].set_xlabel('PC1')
axes[1, 0].set_ylabel('PC2')

# PCA with scaling
axes[1, 1].scatter(X_with_scale[:, 0], X_with_scale[:, 1])
axes[1, 1].set_title(f'PCA with Scaling\nPC1 explains {pca_with_scale.explained_variance_ratio_[0]:.1%}')
axes[1, 1].set_xlabel('PC1')
axes[1, 1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

print("\n⚠️  Without scaling: PC1 captures only the scale difference, not true structure!")
print("✓ With scaling: PC1 and PC2 capture the true variance directions.")

## Part 3: Eigenfaces — PCA on Face Images

Apply PCA to face images. Each component is an "eigenface" (principal component of face variation).

In [ ]:
# Generate synthetic faces (for Colab compatibility)
# In practice, use: fetch_lfw_people(min_face_per_person=70, resize=0.5)
np.random.seed(42)
n_samples = 100
h, w = 50, 50  # Small image size

# Create synthetic "faces" as random images
X_faces = np.random.randn(n_samples, h * w)
# Add some structure: smooth gradients
for i in range(n_samples):
    img = X_faces[i].reshape(h, w)
    # Add gradient patterns
    img += np.linspace(0, 1, h)[:, np.newaxis]
    img += np.linspace(0, 1, w)[np.newaxis, :]
    X_faces[i] = img.flatten()

# Normalize
X_faces = (X_faces - X_faces.mean()) / X_faces.std()

print(f"Face data shape: {X_faces.shape} (100 images, 50×50 pixels)")

# PCA on faces
n_components = 12
pca_faces = PCA(n_components=n_components)
pca_faces.fit(X_faces)

# Visualize eigenfaces
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.ravel()

for i in range(n_components):
    eigenface = pca_faces.components_[i].reshape(h, w)
    axes[i].imshow(eigenface, cmap='gray')
    axes[i].set_title(f'PC{i+1} ({pca_faces.explained_variance_ratio_[i]:.1%})')
    axes[i].axis('off')

plt.suptitle('Eigenfaces: Principal Components of Face Images')
plt.tight_layout()
plt.show()

print(f"\nExplained variance (first 12 components): {pca_faces.explained_variance_ratio_.sum():.1%}")

## Part 4: Reconstruction Quality vs Components

In [ ]:
# Measure reconstruction error
reconstruction_errors = []
n_comp_range = range(1, min(51, X_faces.shape[1]))

for n_comp in n_comp_range:
    pca_temp = PCA(n_components=n_comp)
    X_reduced = pca_temp.fit_transform(X_faces)
    X_reconstructed = pca_temp.inverse_transform(X_reduced)
    error = np.linalg.norm(X_faces - X_reconstructed) / np.linalg.norm(X_faces)
    reconstruction_errors.append(error)

# Visualize reconstruction
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Test image
test_idx = 0
test_img = X_faces[test_idx].reshape(h, w)

# Original
axes[0, 0].imshow(test_img, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# Reconstructions with different K
k_values = [2, 5, 10, 20, 50]
for idx, k in enumerate(k_values[1:]):
    pca_k = PCA(n_components=k)
    pca_k.fit(X_faces)
    X_reduced_k = pca_k.transform(X_faces[[test_idx]])
    X_recon_k = pca_k.inverse_transform(X_reduced_k)
    recon_img = X_recon_k[0].reshape(h, w)
    axes[0, idx + 1].imshow(recon_img, cmap='gray')
    error = np.linalg.norm(test_img.flatten() - recon_img.flatten())
    axes[0, idx + 1].set_title(f'K={k} (error={error:.2f})')
    axes[0, idx + 1].axis('off')

# Error curve
axes[1, 0].semilogy(n_comp_range, reconstruction_errors, 'b-', linewidth=2)
axes[1, 0].set_xlabel('Number of Components')
axes[1, 0].set_ylabel('Relative Reconstruction Error')
axes[1, 0].set_title('Reconstruction Error vs Components')
axes[1, 0].grid(True, alpha=0.3)

# Explained variance
pca_full = PCA()
pca_full.fit(X_faces)
axes[1, 1].plot(np.cumsum(pca_full.explained_variance_ratio_[:50]), 'g-', linewidth=2)
axes[1, 1].set_xlabel('Number of Components')
axes[1, 1].set_ylabel('Cumulative Explained Variance')
axes[1, 1].set_title('Variance Explained')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(0.95, color='r', linestyle='--', label='95% threshold')
axes[1, 1].legend()

# Hide extra subplots
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## Part 5: Kernel PCA for Nonlinear Reduction

Linear PCA fails on nonlinearly separable data. Kernel PCA applies the kernel trick to find nonlinear manifolds.

In [ ]:
# Generate nonlinearly separable data (Swiss roll)
n_samples = 300
t = np.random.uniform(0, 4*np.pi, n_samples)
height = np.random.uniform(-1, 1, n_samples)

X_swiss = np.column_stack([
    t * np.cos(t),
    height,
    t * np.sin(t)
])

# Add noise
X_swiss += np.random.randn(n_samples, 3) * 0.1

# Linear PCA
pca_linear = PCA(n_components=2)
X_linear = pca_linear.fit_transform(X_swiss)

# Kernel PCA
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.1, random_state=42)
X_kpca = kpca.fit_transform(X_swiss)

# Visualize
fig = plt.figure(figsize=(15, 5))

# Original 3D
ax = fig.add_subplot(131, projection='3d')
ax.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=t, cmap='hsv', alpha=0.6)
ax.set_title('Original Swiss Roll (3D)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')

# Linear PCA
ax = fig.add_subplot(132)
scatter = ax.scatter(X_linear[:, 0], X_linear[:, 1], c=t, cmap='hsv', alpha=0.6)
ax.set_title('Linear PCA')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_aspect('equal')

# Kernel PCA
ax = fig.add_subplot(133)
scatter = ax.scatter(X_kpca[:, 0], X_kpca[:, 1], c=t, cmap='hsv', alpha=0.6)
ax.set_title('Kernel PCA (RBF)')
ax.set_xlabel('KPC1')
ax.set_ylabel('KPC2')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("Kernel PCA with RBF kernel preserves the Swiss roll manifold structure!")

## Part 6: PCA as Preprocessing for Classification

In [ ]:
# Use Iris with SVM classifier
iris = load_iris()
X, y = iris.data, iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Try different number of components
n_components_range = range(1, 5)
train_accuracies = []
test_accuracies = []

for n_comp in n_components_range:
    # PCA
    pca = PCA(n_components=n_comp)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    # SVM classifier
    svm = SVC(kernel='rbf')
    svm.fit(X_train_pca, y_train)
    
    train_acc = svm.score(X_train_pca, y_train)
    test_acc = svm.score(X_test_pca, y_test)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    print(f"Components={n_comp}: Train={train_acc:.3f}, Test={test_acc:.3f}")

# Baseline: no PCA
svm_full = SVC(kernel='rbf')
svm_full.fit(X_train_scaled, y_train)
baseline_test = svm_full.score(X_test_scaled, y_test)
print(f"No PCA (full features): Test={baseline_test:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n_components_range, train_accuracies, 'o-', label='Train', linewidth=2, markersize=8)
axes[0].plot(n_components_range, test_accuracies, 's-', label='Test', linewidth=2, markersize=8)
axes[0].axhline(baseline_test, color='r', linestyle='--', label='Full (no PCA)', linewidth=2)
axes[0].set_xlabel('Number of PCA Components')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('SVM Classification Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.8, 1.0])

# Feature importance: explained variance
pca_full = PCA()
pca_full.fit(X_train_scaled)
axes[1].bar(range(1, 5), pca_full.explained_variance_ratio_)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance Ratio')
axes[1].set_title('Variance Explained by Each Component')
axes[1].set_xticks(range(1, 5))

plt.tight_layout()
plt.show()

## Summary

### Key Takeaways
1. **Always standardize** before PCA (unless dimensions are on same scale)
2. **Scree plots** guide component selection
3. **Eigenfaces** show PCA captures meaningful facial variation
4. **Reconstruction error** decreases with more components
5. **Kernel PCA** handles nonlinear manifolds
6. **PCA as preprocessing** often improves downstream classifier performance

### Practical Workflow
1. Standardize features
2. Fit PCA on training data
3. Select K via variance threshold (typically 95%)
4. Transform train and test data
5. Use reduced features for downstream tasks

### Next Steps
- Lesson 6: Manifold Learning (t-SNE, UMAP)
- Lesson 7: Anomaly Detection
- Advanced: Incremental PCA, probabilistic PCA, autoencoders